# 🚀 Notebook Bonus : Pour aller plus loin (Préparation UP 2 & UP 3)
## Du Data Profiling au Data Cleaning & Agrégations

Félicitations pour avoir bouclé le **Cours 1** ! 🎉

Lors des prochaines séances, vous allez être confrontés à de **vrais jeux de données d'entreprise** : bruités, incomplets, avec des doublons et des schémas instables.

Ce notebook facultatif vous propose d'anticiper 3 compétences clés du Data Engineer :
1. **Détecter et purger les doublons** (`drop_duplicates`).
2. **Traiter les valeurs manquantes sans déformer la réalité** (`fillna` avec médiane/mode vs `dropna`).
3. **Calculer vos premières agrégations métier** (`groupby`).

In [ ]:
import pandas as pd
import numpy as np

print("Environnement prêt !")

## 1. Génération du dataset : Logs E-commerce
Exécutez la cellule suivante pour créer un historique de commandes de 5 000 lignes contenant délibérément des pièges réels.

In [ ]:
np.random.seed(101)
n = 5000

categories = ["Informatique", "Audio", "Gaming", "Accessoires"]
villes = ["Lille", "Paris", "Lyon", "Marseille", "Bordeaux", None]

df_raw = pd.DataFrame({
    "order_id": [f"ORD-{i:05d}" for i in range(1, n + 1)],
    "categorie": np.random.choice(categories, size=n, p=[0.3, 0.25, 0.35, 0.1]),
    "prix_ht": np.random.uniform(15.0, 800.0, size=n).round(2),
    "quantite": np.random.randint(1, 5, size=n),
    "ville_livraison": np.random.choice(villes, size=n, p=[0.25, 0.3, 0.15, 0.15, 0.1, 0.05])
})

# Injection de valeurs manquantes (NaN) sur le prix
indices_nan_prix = np.random.choice(df_raw.index, size=150, replace=False)
df_raw.loc[indices_nan_prix, "prix_ht"] = np.nan

# Injection artificielle de doublons bruts (lignes entières recopiées)
doublons = df_raw.sample(80, random_state=42)
df_raw = pd.concat([df_raw, doublons], ignore_index=True)

print(f"Dataset généré : {df_raw.shape[0]} lignes, {df_raw.shape[1]} colonnes.")
df_raw.head()

## 2. Défi 1 : La chasse aux doublons
En pipeline de données, un bug d'ingestion ou un retry HTTP automatique peut provoquer des ré-écritures de lignes identiques.

**Rappel :**
- `df.duplicated().sum()` compte le nombre de lignes en doublon exact.
- `df.drop_duplicates(keep='first')` supprime les doublons en conservant la première occurrence.

### Consignes :
1. Stockez dans `nb_doublons_initial` le nombre total de lignes en doublon dans `df_raw`.
2. Créez un nouveau DataFrame `df_dedup` débarrassé de ces doublons.

In [ ]:
# TODO : Comptez puis supprimez les doublons
nb_doublons_initial = ...
df_dedup = ...

print(f"Doublons détectés : {nb_doublons_initial}")
print(f"Nouvelle dimension après déduplication : {df_dedup.shape}")

In [ ]:
assert nb_doublons_initial == 80, f"Attendu 80 doublons, obtenu {nb_doublons_initial}."
assert df_dedup.shape[0] == 5000, f"df_dedup devrait avoir exactement 5000 lignes, obtenu {df_dedup.shape[0]}."
assert df_dedup.duplicated().sum() == 0, "Il reste des doublons dans df_dedup !"
print("✅ Défi 1 validé : Vos données sont dédupliquées !")

## 3. Défi 2 : Stratégie de traitement des NaN
Supprimer aveuglément les lignes contenant des `NaN` (`dropna()`) est une hérésie si cela détruit 30% du dataset. En Data Engineering, on adopte des stratégies ciblées :
- **Imputation statistique :** Pour une métrique numérique continue (ex. `prix_ht`), remplacer les `NaN` par la **médiane** de la colonne évite de fausser les sommes.
- **Valeur par défaut :** Pour une dimension catégorielle (ex. `ville_livraison`), remplacer les `NaN` par une catégorie explicite `'INCONNU'` permet de conserver la ligne sans casser les filtres.

### Consignes :
À partir de `df_dedup` :
1. Calculez la médiane de `prix_ht` (arrondie à 2 décimales) dans la variable `mediane_prix`.
2. Remplacez les valeurs manquantes de `prix_ht` par cette médiane via `.fillna()`.
3. Remplacez les valeurs manquantes de `ville_livraison` par `'INCONNU'`.
4. Stockez le résultat propre dans `df_clean`.

In [ ]:
# TODO : Imputation des NaN
df_clean = df_dedup.copy()

mediane_prix = ...
df_clean["prix_ht"] = df_clean["prix_ht"].fillna(...)
df_clean["ville_livraison"] = df_clean["ville_livraison"].fillna(...)

print("Médiane utilisée :", mediane_prix)
print("Vérification des NaN résiduels :\n", df_clean.isna().sum())

In [ ]:
assert df_clean["prix_ht"].isna().sum() == 0, "Il reste des NaN dans la colonne prix_ht."
assert df_clean["ville_livraison"].isna().sum() == 0, "Il reste des NaN dans la colonne ville_livraison."
assert "INCONNU" in df_clean["ville_livraison"].values, "La modalité 'INCONNU' n'a pas été ajoutée."
assert df_clean.shape == (5000, 5), "La dimension du DataFrame ne doit pas changer lors d'un fillna !"
print("✅ Défi 2 validé : Votre dataset est propre et complet !")

## 4. Défi 3 : Introduction aux agrégations (`groupby`)
Le `groupby` est le pendant du `GROUP BY` SQL en Pandas : il découpe la donnée en sous-groupes (*Split*), applique une fonction (*Apply*), et recombine le résultat (*Combine*).

Ajoutons d'abord une colonne calculée `montant_total` (chiffre d'affaires de la ligne) :

In [ ]:
df_clean["montant_total"] = df_clean["prix_ht"] * df_clean["quantite"]
df_clean.head(3)

### Consignes :
À partir de `df_clean` :
1. Utilisez `.groupby('categorie')['montant_total'].sum()` pour calculer le chiffre d'affaires total par catégorie.
2. Triez les catégories par chiffre d'affaires décroissant avec `.sort_values(ascending=False)`.
3. Stockez cette Series dans la variable `ca_par_categorie`.
4. Identifiez la catégorie numéro 1 (la plus rentable) dans `top_categorie` (utilisez `.index[0]`).

In [ ]:
# TODO : Agrégation et classement
ca_par_categorie = ...
top_categorie = ...

print("Chiffre d'affaires par catégorie :\n", ca_par_categorie)
print(f"\n🏆 Catégorie n°1 : {top_categorie}")

In [ ]:
assert isinstance(ca_par_categorie, pd.Series), "ca_par_categorie doit être une Series Pandas."
assert len(ca_par_categorie) == 4, "Il doit y avoir 4 catégories distinctes."
assert ca_par_categorie.is_monotonic_decreasing, "La Series doit être triée par ordre décroissant !"
assert top_categorie in ["Gaming", "Informatique"], "Vérifiez le calcul du top catégorie."
print("🎉 FÉLICITATIONS ! Vous avez bouclé les défis d'anticipation avec brio !")
print("Vous êtes prêts pour aborder l'UP 2 et l'UP 3 en toute confiance !")